In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt

In [2]:
names = locals()

In [3]:
country_pseudo_values        = pd.read_csv('../data/country-pseudo-values.csv')
country_raster_50km_         = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')
species_richness_50km        = rio.open('../results/area-of-habitat-50km/species-richness-vertebrates-50km.tif').read(1)
RWR_50km = rio.open('../results/area-of-habitat-50km/rarity-weighted-richness-vertebrates-50km.tif').read(1)
country_raster_50km          = country_raster_50km_.read(1)

In [4]:
def assess_biodiversity_impacts(scenario, level_1='M', level_2='M', level_3='M', res='50km'):
    scale_dict = {'10km': 10, '50km': 50}
    scale = scale_dict[res]
    country_raster_ = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    mining_area_expansion_2050 = rio.open(f'../results/uncertainty-analysis/mining-area-expansion-2050-{scenario}-50km-{level_1}{level_2}{level_3}.tif').read(1)
    RWR = rio.open(f'../results/area-of-habitat-{res}/rarity-weighted-richness-vertebrates-{res}.tif').read(1)
    affected_species_range = (mining_area_expansion_2050 / scale**2) * RWR

    df_results = pd.DataFrame([], index=['CR', 'EN', 'VU', 'NT', 'LC/DD', 'total'], columns=['mammals', 'birds', 'amphibians', 'reptiles', 'vertebrates'])
    for taxon in ['mammals', 'birds', 'amphibians', 'reptiles', 'vertebrates']:
        for category in ['CR', 'EN', 'VU', 'NT', 'total']:
            if category != 'total':
                RWR_taxon = rio.open(f'../results/area-of-habitat-{res}/rarity-weighted-richness-{taxon}-{category}-{res}.tif').read(1)
                affected_species_range_taxon = (mining_area_expansion_2050 / scale**2) * RWR_taxon
                df_results.loc[category, taxon] = np.sum(affected_species_range_taxon)
            else:
                RWR_taxon = rio.open(f'../results/area-of-habitat-{res}/rarity-weighted-richness-{taxon}-{res}.tif').read(1)
                affected_species_range_taxon = (mining_area_expansion_2050 / scale**2) * RWR_taxon
                df_results.loc['total', taxon] = np.sum(affected_species_range_taxon)

    df_results.loc['LC/DD'] = df_results.loc['total'] - df_results.loc[['CR', 'EN', 'VU', 'NT']].sum()

    df_results = df_results.loc[['LC/DD','NT', 'VU', 'EN', 'CR'], ['mammals', 'birds', 'amphibians', 'reptiles']]

    output_path = f'../results/uncertainty-analysis/affected-species-range-2050-{scenario}-{res}-{level_1}{level_2}{level_3}.tif'
    with rio.open(
        output_path,
        "w",
        driver="GTiff",
        height=country_raster_.shape[0],
        width=country_raster_.shape[1],
        count=1,
        dtype=affected_species_range.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform,
        compress='lzw'
    ) as dst:
        dst.write(affected_species_range, 1)

    df_results.to_csv(f'../results/uncertainty-analysis/affected-species-range-2050-{scenario}-{level_1}{level_2}{level_3}.csv')

    return affected_species_range, df_results

In [ ]:
for scenario in ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE', 'CON', 'IAP']:
    for L_1 in ['L', 'M', 'H']:
        for L_2 in ['L', 'M', 'H']:
            for L_3 in ['L', 'M', 'H']:
                gg, hh = assess_biodiversity_impacts(scenario, level_1=L_1, level_2=L_2, level_3=L_3)
                print(f'{scenario}-{L_1}{L_2}{L_3}', np.sum(gg))